<!-- paired-role-banner -->
> 🟩 **정답 노트북 06 · 참고 코드 창**
>
> 왼쪽 실습 노트북과 같은 셀 순서입니다. 결과만 복사하기보다 각 shape, 자료형, 손실값과
> 설계 이유를 먼저 예측한 뒤 필요한 부분만 확인하세요.

# 06. PyTorch 텍스트 분류 파이프라인

고객지원 티켓을 예제로 **데이터 로딩부터 오분류 분석까지** 직접 구현합니다. 프레임워크가 숨겨 주는 단계를 의도적으로 펼쳐 놓았기 때문에, 각 tensor의 shape를 추적하며 실행하세요.

예상 시간: 90~120분 · CUDA → MPS → CPU 자동 선택 · 결과물: 작은 분류 checkpoint

## 전체 흐름

`CSV → group-safe split → tokenizer → vocabulary → Dataset → collate/padding → DataLoader → model → loss/backward → checkpoint → error analysis`

현업에서 특히 중요한 두 가지를 함께 연습합니다.

- 같은 사건의 변형 문장이 train/validation 양쪽에 들어가는 **데이터 누수 방지**
- 모델 가중치뿐 아니라 vocabulary와 label mapping까지 저장하는 **재현 가능한 checkpoint**

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
import random
import re

import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    None,
)
assert PROJECT_ROOT is not None, "프로젝트 루트에서 JupyterLab을 실행해 주세요."
DATA_PATH = PROJECT_ROOT / "data" / "customer_support_tickets.csv"
print(ACCELERATOR.summary())
print("data:", DATA_PATH)

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.read_csv -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.read_csv(filepath_or_buffer, ...)</code></summary>

#### `pd.read_csv(filepath_or_buffer, ...)`

- **역할:** CSV 표 데이터를 DataFrame으로 읽습니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
tickets = pd.read_csv(DATA_PATH)
required = {"ticket_id", "case_id", "text", "label"}
missing = required - set(tickets.columns)
assert not missing, f"필수 열 누락: {missing}"
assert tickets["ticket_id"].is_unique
assert tickets[["case_id", "text", "label"]].notna().all().all()

print("shape:", tickets.shape)
print("labels:", sorted(tickets["label"].unique()))
tickets[["ticket_id", "case_id", "text", "label"]].head(3)

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataFrame.groupby -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>frame.groupby(by, ...)</code></summary>

#### `frame.groupby(by, ...)`

- **역할:** key가 같은 행을 그룹으로 묶습니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
label_summary = (
    tickets.groupby("label")
    .agg(rows=("ticket_id", "size"), cases=("case_id", "nunique"))
    .sort_index()
)
lengths = tickets["text"].str.len()
print(label_summary)
print("\n문자 길이 요약:")
print(lengths.describe(percentiles=[0.5, 0.9]).round(1))

# 한 case_id에 서로 다른 label이 섞이면 group split 전에 수정해야 합니다.
labels_per_case = tickets.groupby("case_id")["label"].nunique()
assert labels_per_case.max() == 1

## 1. Group-safe train/validation split

이 데이터에는 한 사건(`case_id`)을 다르게 표현한 문장이 여러 개 있습니다. 행을 무작위 분할하면 거의 같은 문장이 양쪽에 들어가 검증 점수가 부풀 수 있습니다.

따라서 label별 case 목록을 나눈 뒤 해당 case의 모든 행을 한쪽에 배치합니다.

In [ ]:
rng = random.Random(SEED)
validation_cases = set()

for label, group in tickets.groupby("label"):
    case_ids = sorted(group["case_id"].unique())
    rng.shuffle(case_ids)
    valid_count = max(1, round(len(case_ids) * 0.2))
    validation_cases.update(case_ids[:valid_count])

is_validation = tickets["case_id"].isin(validation_cases)
train_df = tickets.loc[~is_validation].reset_index(drop=True)
valid_df = tickets.loc[is_validation].reset_index(drop=True)

assert set(train_df["case_id"]).isdisjoint(valid_df["case_id"])
assert set(train_df["label"]) == set(valid_df["label"])
print("train/valid rows:", len(train_df), len(valid_df))
print("train/valid cases:", train_df.case_id.nunique(), valid_df.case_id.nunique())

## 2. 투명한 tokenizer

교육용으로 한글은 음절 단위, 영문/숫자는 묶음 단위로 나눕니다. 실무의 subword tokenizer보다 단순하지만 OOV, truncation, padding 개념을 관찰하기 좋습니다.

tokenizer는 train 데이터만 보고 vocabulary를 만들어야 합니다. validation을 포함해 vocabulary를 만들면 작은 형태의 데이터 누수가 됩니다.

In [ ]:
TOKEN_PATTERN = re.compile(r"[가-힣]|[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:[.,]\d+)?|[^\w\s]")


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.strip().lower())


sample_text = "API 429 오류가 나고 카드가 두 번 결제됐어요!"
sample_tokens = tokenize(sample_text)
print(sample_tokens)
assert "api" in sample_tokens and "429" in sample_tokens

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]


@dataclass
class Vocabulary:
    itos: list[str]

    def __post_init__(self):
        self.stoi = {token: index for index, token in enumerate(self.itos)}

    @classmethod
    def build(cls, texts, min_frequency: int = 1):
        counts = Counter(token for text in texts for token in tokenize(text))
        regular = sorted(
            (token for token, count in counts.items() if count >= min_frequency),
            key=lambda token: (-counts[token], token),
        )
        return cls(SPECIAL_TOKENS + regular)

    def __len__(self):
        return len(self.itos)

    def encode(self, text: str, max_length: int = 80) -> list[int]:
        ids = [self.stoi["<bos>"]]
        ids += [self.stoi.get(token, self.stoi["<unk>"]) for token in tokenize(text)]
        ids += [self.stoi["<eos>"]]
        ids = ids[:max_length]
        ids[-1] = self.stoi["<eos>"]
        return ids

    def decode(self, ids) -> str:
        return " ".join(self.itos[int(index)] for index in ids)

In [ ]:
vocabulary = Vocabulary.build(train_df["text"], min_frequency=1)
encoded = vocabulary.encode("배송 API 오류 999")

print("vocab size:", len(vocabulary))
print("special ids:", {token: vocabulary.stoi[token] for token in SPECIAL_TOKENS})
print("encoded:", encoded)
print("decoded:", vocabulary.decode(encoded))
print("OOV count:", encoded.count(vocabulary.stoi["<unk>"]))
assert vocabulary.stoi["<pad>"] == 0

## 3. Dataset: 한 샘플을 tensor로 바꾸기

`Dataset.__getitem__`은 한 샘플만 반환합니다. 아직 padding하지 않는 이유는 배치마다 가장 긴 길이에 맞춰 padding해야 불필요한 계산을 줄일 수 있기 때문입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame,torch.tensor -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
label_names = sorted(train_df["label"].unique())
label_to_id = {label: index for index, label in enumerate(label_names)}
id_to_label = {index: label for label, index in label_to_id.items()}


class TicketDataset(Dataset):
    def __init__(
        self, frame: pd.DataFrame, vocabulary: Vocabulary, max_length: int = 80
    ):
        self.frame = frame.reset_index(drop=True)
        self.vocabulary = vocabulary
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {
            "input_ids": torch.tensor(
                self.vocabulary.encode(row["text"], self.max_length), dtype=torch.long
            ),
            "label": torch.tensor(label_to_id[row["label"]], dtype=torch.long),
            "ticket_id": row["ticket_id"],
            "text": row["text"],
        }


train_dataset = TicketDataset(train_df, vocabulary)
valid_dataset = TicketDataset(valid_df, vocabulary)

In [ ]:
item = train_dataset[0]
print("keys:", item.keys())
print("input shape/dtype:", item["input_ids"].shape, item["input_ids"].dtype)
print("label:", item["label"].item(), id_to_label[item["label"].item()])
print("text:", item["text"])
assert item["input_ids"].ndim == 1
assert item["label"].ndim == 0

## 4. Collate: 가변 길이를 한 배치로 묶기

`DataLoader`가 샘플 목록을 `collate_fn`에 전달합니다. 여기서 `[T]` 여러 개를 `[B,T_max]`로 padding하고, 실제 토큰 위치가 1인 `attention_mask`를 만듭니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.stack,ACCELERATOR.move,nn.Embedding,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
PAD_ID = vocabulary.stoi["<pad>"]


def collate_tickets(items):
    input_ids = pad_sequence(
        [item["input_ids"] for item in items],
        batch_first=True,
        padding_value=PAD_ID,
    )
    attention_mask = input_ids.ne(PAD_ID)
    labels = torch.stack([item["label"] for item in items])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "ticket_ids": [item["ticket_id"] for item in items],
        "texts": [item["text"] for item in items],
    }


demo_batch = collate_tickets([train_dataset[0], train_dataset[1], train_dataset[2]])
demo_batch = ACCELERATOR.move(demo_batch)
for key in ["input_ids", "attention_mask", "labels"]:
    print(key, demo_batch[key].shape, demo_batch[key].dtype)
assert demo_batch["input_ids"].shape == demo_batch["attention_mask"].shape
assert demo_batch["labels"].shape == (3,)

# padding을 제외한 mean pooling을 손으로 계산합니다.
demo_embedding = nn.Embedding(len(vocabulary), 16, padding_idx=PAD_ID).to(DEVICE)
embedded = demo_embedding(demo_batch["input_ids"])  # [B, T, D]
weights = demo_batch["attention_mask"].unsqueeze(-1).float()  # [B, T, 1]
pooled = (embedded * weights).sum(1) / weights.sum(1).clamp_min(1.0)

print("embedded:", embedded.shape)
print("weights: ", weights.shape)
print("pooled:  ", pooled.shape)
assert pooled.shape == (3, 16)

## 5. 분류 모델

구조는 `Embedding → masked mean pooling → LayerNorm → MLP → logits`입니다. 단순하지만 LLM 파이프라인의 입력 ID, mask, embedding, logits 개념을 그대로 포함합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.LayerNorm,nn.Linear,nn.GELU,nn.Dropout -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

<details>
<summary><code>nn.Dropout(p=0.5)</code></summary>

#### `nn.Dropout(p=0.5)`

- **역할:** 학습 중 일부 activation을 무작위로 0으로 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\frac{m_i}{1-p}x_i,\qquad m_i\sim\operatorname{Bernoulli}(1-p)
$$

- **기호:** $p$는 제거 확률, $m_i$는 학습 중 표본화한 mask입니다.
- **수식 → 코드:** `model.train()`일 때 mask와 $1/(1-p)$ scale을 적용하고 `eval()`에서는 그대로 둡니다.
- **직관:** 일부 activation에 의존하지 못하게 하면서 기댓값은 유지합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass
class ClassifierConfig:
    vocab_size: int
    num_classes: int
    pad_id: int
    embedding_dim: int = 32
    hidden_dim: int = 48
    dropout: float = 0.15


class MeanPoolClassifier(nn.Module):
    def __init__(self, config: ClassifierConfig):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(
            config.vocab_size, config.embedding_dim, padding_idx=config.pad_id
        )
        self.head = nn.Sequential(
            nn.LayerNorm(config.embedding_dim),
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.num_classes),
        )

    def forward(self, input_ids, attention_mask):
        assert input_ids.ndim == attention_mask.ndim == 2
        x = self.embedding(input_ids)  # [B, T, D]
        weights = attention_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        logits = self.head(pooled)
        assert logits.shape == (input_ids.shape[0], self.config.num_classes)
        return logits

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.softmax,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.softmax(dim, dtype=None)</code></summary>

#### `tensor.softmax(dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** method 문법으로 attention score나 class logit을 정규화합니다.
- **주의점:** `dim`이 확률 합을 낼 축이며 입력은 일반적으로 raw logit이어야 합니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [Tensor.softmax](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.softmax.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
config = ClassifierConfig(len(vocabulary), len(label_names), PAD_ID)
model = MeanPoolClassifier(config).to(DEVICE)
logits = model(demo_batch["input_ids"], demo_batch["attention_mask"])
probabilities = logits.softmax(dim=-1)

print("logits:", logits.shape)
print("first probabilities:", probabilities[0])
print("probability sum:", probabilities[0].sum().item())
assert torch.allclose(
    probabilities.sum(-1), torch.ones_like(probabilities.sum(-1)), atol=1e-6
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.AdamW,Module.parameters,Tensor.detach,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,clip_grad_norm_,Optimizer.step,Tensor.item,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)</code></summary>

#### `torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)`

- **역할:** weight decay를 gradient update와 분리한 AdamW optimizer를 만듭니다.
- **입력·반환:** 파라미터와 hyperparameter를 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer 계열에서 흔한 최적화 규칙을 재현합니다.
- **주의점:** bias·정규화 파라미터를 decay에서 제외하는 설정도 자주 사용합니다.

##### 관련 수식과 코드 연결

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

- **기호:** $\lambda$는 weight decay이고 나머지 기호는 Adam과 같습니다.
- **수식 → 코드:** 코드의 `weight_decay`가 gradient 기반 Adam update와 분리된 축소 항으로 적용됩니다.
- **직관:** 손실 gradient와 무관하게 큰 weight를 조금씩 줄이는 것이 Adam과의 핵심 차이입니다.
- **shape 확인:** 파라미터와 모멘트 tensor의 shape은 변하지 않습니다.
- **공식 문서:** [torch.optim.AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)</code></summary>

#### `clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)`

- **역할:** 전체 gradient norm이 한계를 넘으면 같은 비율로 축소합니다.
- **입력·반환:** 파라미터와 최대 norm을 받고 clipping 전 전체 norm을 반환합니다.
- **이 셀에서 쓰는 이유:** RNN·Transformer·RL 학습의 폭주하는 update를 제한합니다.
- **주의점:** `backward()` 뒤 `step()` 전에 호출해야 하며 근본 원인을 없애지는 않습니다.

##### 관련 수식과 코드 연결

$$
g' = g\cdot\min\!\left(1,\frac{c}{\lVert g\rVert_p+\epsilon}\right)
$$

- **기호:** $g$는 모든 parameter gradient를 합친 벡터, $c$는 `max_norm`입니다.
- **수식 → 코드:** 전체 norm이 한계를 넘을 때만 모든 gradient를 같은 비율로 줄입니다.
- **직관:** 방향은 유지하면서 update 크기만 제한합니다.
- **shape 확인:** 각 parameter의 gradient shape은 유지됩니다.
- **공식 문서:** [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# 한 번의 update를 분해해 확인합니다.
one_step_model = MeanPoolClassifier(config).to(DEVICE)
one_step_optimizer = torch.optim.AdamW(one_step_model.parameters(), lr=3e-3)
before = one_step_model.head[-1].weight.detach().clone()

one_step_optimizer.zero_grad(set_to_none=True)
one_step_logits = one_step_model(demo_batch["input_ids"], demo_batch["attention_mask"])
one_step_loss = nn.functional.cross_entropy(one_step_logits, demo_batch["labels"])
one_step_loss.backward()
gradient_norm = nn.utils.clip_grad_norm_(one_step_model.parameters(), max_norm=1.0)
one_step_optimizer.step()

after = one_step_model.head[-1].weight.detach()
print("loss:", one_step_loss.item(), "gradient norm:", float(gradient_norm))
print("parameter changed:", not torch.equal(before, after))
assert not torch.equal(before, after)

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.train,torch.inference_mode,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, predictions, targets = 0.0, [], []

    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for batch in loader:
            batch = ACCELERATOR.move(batch)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss = nn.functional.cross_entropy(logits, batch["labels"])
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(batch["labels"])
            predictions.extend(logits.argmax(-1).cpu().tolist())
            targets.extend(batch["labels"].cpu().tolist())

    return {
        "loss": total_loss / len(targets),
        "accuracy": sum(p == y for p, y in zip(predictions, targets)) / len(targets),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
    }

<!-- api-explanations:start -->
<!-- api-explanations:keys=DataLoader -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>DataLoader(dataset, batch_size=1, shuffle=False, ...)</code></summary>

#### `DataLoader(dataset, batch_size=1, shuffle=False, ...)`

- **역할:** dataset에서 mini-batch를 반복 생성합니다.
- **입력·반환:** dataset과 batching 설정을 받고 iterable loader를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 데이터를 batch 단위로 섞고 모델에 공급합니다.
- **주의점:** Windows 다중 worker는 진입점 보호가 필요하며 마지막 batch 크기도 확인합니다.
- **공식 문서:** [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_tickets,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_tickets,
    num_workers=0,
)

torch.manual_seed(SEED)
model = MeanPoolClassifier(config).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)
print("parameters:", sum(parameter.numel() for parameter in model.parameters()))

In [ ]:
history = []
best_state = None
best_f1 = -1.0

for epoch in range(1, 7):
    train_metrics = run_epoch(model, train_loader, optimizer)
    valid_metrics = run_epoch(model, valid_loader)
    history.append(
        {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"valid_{k}": v for k, v in valid_metrics.items()},
        }
    )
    if valid_metrics["macro_f1"] > best_f1:
        best_f1 = valid_metrics["macro_f1"]
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
    print(
        f"epoch={epoch} train_loss={train_metrics['loss']:.3f} "
        f"valid_loss={valid_metrics['loss']:.3f} "
        f"valid_f1={valid_metrics['macro_f1']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.round(3)

## 6. Checkpoint 저장

state dict만 저장하면 추론 시 vocabulary와 label 순서를 잃을 수 있습니다. 최소한 다음을 함께 저장합니다.

- model state와 모델 config
- token 순서(`itos`)와 label 순서
- seed, 학습 epoch, 검증 지표 같은 provenance

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.eval -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
CHECKPOINT_PATH = PROJECT_ROOT / "artifacts" / "notebook_text_classifier_solution.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
model.load_state_dict(best_state)

checkpoint = {
    "model_state": model.state_dict(),
    "model_config": asdict(config),
    "vocabulary": vocabulary.itos,
    "label_names": label_names,
    "seed": SEED,
    "best_validation_f1": best_f1,
}
torch.save(checkpoint, CHECKPOINT_PATH)
print("saved:", CHECKPOINT_PATH)
print("size (KB):", round(CHECKPOINT_PATH.stat().st_size / 1024, 1))

# 새 Python 프로세스에서도 같은 방식으로 복원할 수 있는지 확인합니다.
restored_payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
restored_vocab = Vocabulary(restored_payload["vocabulary"])
restored_config = ClassifierConfig(**restored_payload["model_config"])
restored_model = MeanPoolClassifier(restored_config).to(DEVICE)
restored_model.load_state_dict(restored_payload["model_state"])
restored_model.eval()

assert restored_vocab.itos == vocabulary.itos
assert restored_payload["label_names"] == label_names
print("restored best F1:", restored_payload["best_validation_f1"])

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.cpu -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@torch.inference_mode()
def predict_text(text: str, model=restored_model, vocab=restored_vocab):
    ids = torch.tensor(vocab.encode(text), dtype=torch.long, device=DEVICE).unsqueeze(0)
    mask = ids.ne(vocab.stoi["<pad>"])
    probabilities = model(ids, mask).softmax(-1).squeeze(0)
    ranked = probabilities.argsort(descending=True)
    return [
        (label_names[index], float(probabilities[index]))
        for index in ranked.cpu().tolist()
    ]


for text in [
    "카드 결제가 두 번 되었어요",
    "로그인이 안 되고 비밀번호 재설정도 실패합니다",
    "API에서 429 오류가 납니다",
]:
    print("\n", text)
    print(predict_text(text)[:3])

## 7. Error analysis

전체 점수 하나만 보면 개선 방향을 알기 어렵습니다. 틀린 샘플의 정답, 예측, confidence, 텍스트를 한 테이블로 모읍니다. confidence가 높으면서 틀린 사례는 특히 우선적으로 봅니다.

In [ ]:
restored_model.eval()
error_rows = []
all_rows = []

with torch.inference_mode():
    for batch in valid_loader:
        batch = ACCELERATOR.move(batch)
        logits = restored_model(batch["input_ids"], batch["attention_mask"])
        probabilities = logits.softmax(-1)
        confidence, predicted_ids = probabilities.max(-1)
        for ticket_id, text, target_id, predicted_id, score in zip(
            batch["ticket_ids"],
            batch["texts"],
            batch["labels"],
            predicted_ids,
            confidence,
        ):
            row = {
                "ticket_id": ticket_id,
                "truth": id_to_label[int(target_id)],
                "prediction": id_to_label[int(predicted_id)],
                "confidence": float(score),
                "text": text,
            }
            all_rows.append(row)
            if row["truth"] != row["prediction"]:
                error_rows.append(row)

errors = pd.DataFrame(error_rows).sort_values("confidence", ascending=False)
print("validation rows:", len(all_rows), "errors:", len(errors))
errors.head(10)

predictions_df = pd.DataFrame(all_rows)
confusion = pd.crosstab(
    predictions_df["truth"],
    predictions_df["prediction"],
    rownames=["truth"],
    colnames=["prediction"],
    dropna=False,
).reindex(index=label_names, columns=label_names, fill_value=0)
confusion

## 8. 따라 치기 TODO

새 셀을 만들어 다음을 직접 구현하세요.

1. `min_frequency=2`로 vocabulary를 다시 만들고 OOV 비율과 F1을 비교합니다.
2. `mean pooling`을 `max pooling`으로 교체합니다. padding 위치는 매우 작은 값으로 가려야 합니다.
3. embedding 차원을 `16, 32, 64`로 바꾸고 parameter 수·학습 시간·F1을 표로 만듭니다.
4. label별 precision/recall을 직접 계산해 confusion table과 검산합니다.
5. 체크포인트에 데이터 파일의 SHA-256을 넣어 어떤 데이터로 학습했는지 추적합니다.

**도전 과제:** 같은 `case_id`를 무시한 무작위 행 분할을 만들어 점수가 얼마나 달라지는지 확인하고, 왜 그 점수를 신뢰하면 안 되는지 한 문단으로 설명하세요.

## 현업 체크리스트

- split 전에 중복·그룹·시간 순서를 확인한다.
- tokenizer/vocabulary는 train 데이터로만 fit한다.
- `Dataset`은 한 샘플, `collate_fn`은 배치 구성을 책임지게 한다.
- 모든 모델 경계에 shape와 dtype 계약을 둔다.
- validation에서는 `eval()`과 `inference_mode()`를 사용한다.
- checkpoint에 전처리 상태, label mapping, config, seed를 함께 저장한다.
- 평균 지표뿐 아니라 label별 지표와 실제 오류 문장을 검토한다.

다음 노트북에서는 PyTorch 모듈에 기대기 전에 MSE, linear/logistic regression, MLP의 핵심 계산을 더 낮은 수준에서 직접 구현합니다.